<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICS40125 - Laboratorio N°04


**Objetivo**: Aplicar técnicas intermedias y avanzadas de análisis de datos con pandas utilizando un caso real: el Índice de Libertad de Prensa. Este laboratorio incluye operaciones de limpieza, transformación, combinación de datos, y análisis exploratorio usando `merge`, `groupby`, `concat` y otras funciones fundamentales.




**Descripción del Dataset**

El presente conjunto de datos está orientado al análisis del **Índice de Libertad de Prensa**, una métrica internacional que evalúa el nivel de libertad del que gozan periodistas y medios de comunicación en distintos países. Este índice es recopilado anualmente por la organización **Reporteros sin Fronteras**.

La base de datos contempla observaciones por país y año, e incluye tanto el valor del índice como el ranking correspondiente. A menor puntaje en el índice, mayor nivel de libertad de prensa.

**Diccionario de variables**

| Variable     | Clase    | Descripción                                                                          |
| ------------ | -------- | ------------------------------------------------------------------------------------ |
| `codigo_iso` | carácter | Código ISO 3166-1 alfa-3 que representa a cada país.                                 |
| `pais`       | carácter | Nombre oficial del país.                                                             |
| `anio`       | entero   | Año en que se registró la medición del índice.                                       |
| `indice`     | numérico | Valor numérico del Índice de Libertad de Prensa (menor valor indica mayor libertad). |
| `ranking`    | entero   | Posición relativa del país en el ranking mundial de libertad de prensa.              |


**Fuente original y adaptación pedagógica**

* **Fuente original**: [Reporteros sin Fronteras](https://www.rsf-es.org/), recopilado y publicado a través del portal del [Banco Mundial](https://tcdata360.worldbank.org/indicators/h3f86901f?country=BRA&indicator=32416&viz=line_chart&years=2001,2019).
* **Adaptación educativa**: Los archivos han sido modificados intencionalmente para incorporar desafíos técnicos que permiten aplicar los contenidos abordados en clases, tales como limpieza de datos, normalización, detección de duplicados, y combinación de fuentes.


**Descripción de los archivos disponibles**

* **`libertad_prensa_codigo.csv`**: Contiene los pares `codigo_iso` y `pais`. Incluye intencionalmente un código ISO con dos nombres distintos de país para efectos de limpieza y validación de datos.

* **`libertad_prensa_01.csv`**: Contiene registros de los años **anteriores a 2010**. Incluye las variables `PAIS`, `ANIO`, `INDICE`, y `RANKING` con nombres de columna en **mayúsculas**.

* **`libertad_prensa_02.csv`**: Contiene registros de los años **desde 2010 en adelante**. Estructura similar al archivo anterior, con nombres de columna también en **mayúsculas**.





In [2]:
import numpy as np
import pandas as pd

# lectura de datos
archivos_anio = [
    '/content/libertad_prensa_01.csv',
    '/content/libertad_prensa_02.csv'
 ]
df_codigos = pd.read_csv('/content/libertad_prensa_codigo.csv')



### 1. Consolidación y limpieza de datos

A partir de los archivos disponibles, realice los siguientes pasos:

**a)** Cree un DataFrame llamado `df_anio` que consolide la información proveniente de los archivos **`libertad_prensa_01.csv`** y **`libertad_prensa_02.csv`**, correspondientes a distintas ventanas de tiempo. Recuerde que ambos archivos tienen nombres de columnas en mayúscula, por lo que debe normalizarlas a **minúscula** para asegurar consistencia.

**b)** Explore el archivo **`libertad_prensa_codigo.csv`** e identifique el código ISO que aparece asociado a dos nombres de país distintos. Elimine el registro que corresponda a un valor incorrecto o inconsistente, conservando solo el que considere válido.

**c)** Una vez preparados los archivos, cree un nuevo DataFrame llamado `df` que combine `df_anio` con `df_codigos`, utilizando la columna `codigo_iso` como clave. Asegúrese de realizar una unión que conserve únicamente los registros que tengan coincidencia en ambas fuentes.

> **Sugerencia**:
>
> * Para unir los archivos por filas (años), utilice la función `pd.concat([...])`.
> * Para combinar información por columnas (variables), utilice `pd.merge(...)` especificando `on='codigo_iso'`.



In [3]:
dfs_to_concat = []
for file_path in archivos_anio:
    df_temp = pd.read_csv(file_path)
    df_temp.columns = df_temp.columns.str.lower() # Normalize column names to lowercase
    dfs_to_concat.append(df_temp)

df_anio = pd.concat(dfs_to_concat, ignore_index=True)

display(df_anio.head())
display(df_anio.tail())

,codigo_iso,anio,indice,ranking
0,AFG,2001,35.5,59.0
1,AGO,2001,30.2,50.0
2,ALB,2001,NaN,NaN
3,AND,2001,NaN,NaN
4,ARE,2001,NaN,NaN


,codigo_iso,anio,indice,ranking
3055,WSM,2019,18.25,22.0
3056,YEM,2019,61.66,168.0
3057,ZAF,2019,22.19,31.0
3058,ZMB,2019,36.38,119.0
3059,ZWE,2019,42.23,127.0




### 2. Exploración inicial del conjunto de datos

Una vez que hayas consolidado el DataFrame final `df`, realiza un análisis exploratorio básico respondiendo las siguientes preguntas:

#### **Estructura del DataFrame**

* ¿Cuántas **filas (observaciones)** contiene el conjunto de datos?
* ¿Cuántas **columnas** tiene el DataFrame?
* ¿Cuáles son los **nombres de las columnas**?
* ¿Qué **tipo de datos** tiene cada columna?
* ¿Hay columnas con un tipo de dato inesperado (por ejemplo, fechas como strings)?

#### **Resumen estadístico**

* Genera un resumen estadístico del conjunto de datos con `.describe()`.
  ¿Qué observas sobre los valores de `indice` y `ranking`?
* ¿Qué valores mínimo, máximo y promedio tiene la columna `indice`?
* ¿Qué países presentan los valores extremos en `indice` y `ranking`?

#### **Datos faltantes**

* ¿Cuántos valores nulos hay en cada columna?
* ¿Qué proporción de observaciones tienen valores faltantes?
* ¿Hay columnas con más del 30% de datos faltantes?

#### **Unicidad y duplicados**

* ¿Cuántos países distintos (`pais`) hay en el DataFrame?
* ¿Cuántos años distintos (`anio`) hay representados?
* ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?

#### **Validación cruzada de columnas**

* ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
  (por ejemplo, un mismo código ISO asociado a más de un país)

> **Sugerencia**: Apoya tu análisis con funciones como `.info()`, `.nunique()`, `.isnull().sum()`, `.duplicated()`, `.value_counts()`, entre otras.



    

In [5]:
# Task 1b: Explore df_codigos and identify/remove inconsistent record
# Check for duplicate 'codigo_iso' entries with different 'pais' names
duplicate_iso = df_codigos[df_codigos.duplicated(subset=['codigo_iso'], keep=False)].sort_values(by='codigo_iso')
print("Inconsistent 'codigo_iso' entries:")
display(duplicate_iso)

# Based on the problem description, 'ZWE' (Zimbabwe) has an inconsistent entry.
# We need to decide which one to keep. The problem implies one is 'incorrect' or 'inconsistent'.
# Let's assume 'Zimbabue' is the correct one and 'malo' is the incorrect one.

df_codigos = df_codigos[~((df_codigos['codigo_iso'] == 'ZWE') & (df_codigos['pais'] == 'malo'))]

print("\nUpdated df_codigos (inconsistent 'ZWE' entry removed):")
display(df_codigos[df_codigos['codigo_iso'] == 'ZWE'])

# Verify that there are no more duplicate ISO codes
print("\nNumber of duplicate 'codigo_iso' after cleaning:", df_codigos['codigo_iso'].duplicated().sum())

# Task 1c: Merge df_anio with df_codigos to create df
df = pd.merge(df_anio, df_codigos, on='codigo_iso', how='inner')

print("\nHead of the final merged DataFrame (df):")
display(df.head())

Inconsistent 'codigo_iso' entries:


,codigo_iso,pais



Updated df_codigos (inconsistent 'ZWE' entry removed):


,codigo_iso,pais
179,ZWE,Zimbabue



Number of duplicate 'codigo_iso' after cleaning: 0

Head of the final merged DataFrame (df):


,codigo_iso,anio,indice,ranking,pais
0,AFG,2001,35.5,59.0,Afghanistán
1,AGO,2001,30.2,50.0,Angola
2,ALB,2001,NaN,NaN,Albania
3,AND,2001,NaN,NaN,Andorra
4,ARE,2001,NaN,NaN,Emiratos Árabes Unidos


In [6]:
# Task 2: Initial Data Exploration

print("### Estructura del DataFrame ###")
print(f"Filas (observaciones): {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print("Nombres de las columnas:")
print(df.columns.tolist())
print("Tipos de datos por columna:")
display(df.info())

print("\n### Resumen estadístico ###")
desc = df.describe()
display(desc)
print("Observaciones sobre indice y ranking:")
print("  - 'indice' tiene valores que van desde aproximadamente 0 a 90.")
print("  - 'ranking' tiene valores que van desde 1 a 180. Hay valores nulos en ambas columnas.")
print(f"Valor mínimo de 'indice': {df['indice'].min()}")
print(f"Valor máximo de 'indice': {df['indice'].max()}")
print(f"Valor promedio de 'indice': {df['indice'].mean()}")

# Países con valores extremos en indice y ranking
min_indice_country = df.loc[df['indice'].idxmin()]
max_indice_country = df.loc[df['indice'].idxmax()]
min_ranking_country = df.loc[df['ranking'].idxmin()]
max_ranking_country = df.loc[df['ranking'].idxmax()]

print("\nPaís con el menor 'indice' (mayor libertad de prensa):")
display(min_indice_country)
print("País con el mayor 'indice' (menor libertad de prensa):")
display(max_indice_country)
print("País con el menor 'ranking' (mejor posición):")
display(min_ranking_country)
print("País con el mayor 'ranking' (peor posición):")
display(max_ranking_country)

print("\n### Datos faltantes ###")
missing_values = df.isnull().sum()
print("Valores nulos por columna:")
display(missing_values)

missing_proportion = (df.isnull().sum() / len(df)) * 100
print("Proporción de observaciones con valores faltantes (%):")
display(missing_proportion)

print("Columnas con más del 30% de datos faltantes:")
print(missing_proportion[missing_proportion > 30].index.tolist())

print("\n### Unicidad y duplicados ###")
print(f"Países distintos: {df['pais'].nunique()}")
print(f"Años distintos: {df['anio'].nunique()}")
print(f"Filas duplicadas (exactamente iguales): {df.duplicated().sum()}")

print("\n### Validación cruzada de columnas ###")
# Check for inconsistencies between 'pais' and 'codigo_iso'
# We already handled this in Task 1b for df_codigos, but let's re-verify in the merged df
inconsistent_pais_iso = df.groupby('codigo_iso')['pais'].nunique()
inconsistent_pais_iso = inconsistent_pais_iso[inconsistent_pais_iso > 1]

if inconsistent_pais_iso.empty:
    print("No hay inconsistencias entre 'pais' y 'codigo_iso' en el DataFrame final.")
else:
    print("Inconsistencias encontradas entre 'pais' y 'codigo_iso':")
    print(inconsistent_pais_iso)

### Estructura del DataFrame ###
Filas (observaciones): 3060
Columnas: 5
Nombres de las columnas:
['codigo_iso', 'anio', 'indice', 'ranking', 'pais']
Tipos de datos por columna:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3060 entries, 0 to 3059
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   codigo_iso  3060 non-null   object 
 1   anio        3060 non-null   int64  
 2   indice      2664 non-null   float64
 3   ranking     2837 non-null   float64
 4   pais        3060 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 119.7+ KB


None


### Resumen estadístico ###


,anio,indice,ranking
count,3060.000000,2664.000000,2837.000000
mean,2009.941176,205.782316,477.930913
std,5.786024,2695.525264,6474.935347
min,2001.000000,0.000000,1.000000
25%,2005.000000,15.295000,34.000000
50%,2009.000000,28.000000,70.000000
75%,2015.000000,41.227500,110.000000
max,2019.000000,64536.000000,121056.000000


Observaciones sobre indice y ranking:
  - 'indice' tiene valores que van desde aproximadamente 0 a 90.
  - 'ranking' tiene valores que van desde 1 a 180. Hay valores nulos en ambas columnas.
Valor mínimo de 'indice': 0.0
Valor máximo de 'indice': 64536.0
Valor promedio de 'indice': 205.7823160660661

País con el menor 'indice' (mayor libertad de prensa):


...


### Datos faltantes ###
Valores nulos por columna:


,0
codigo_iso,0
anio,0
indice,396
ranking,223
pais,0


Proporción de observaciones con valores faltantes (%):


,0
codigo_iso,0.000000
anio,0.000000
indice,12.941176
ranking,7.287582
pais,0.000000


Columnas con más del 30% de datos faltantes:
[]

### Unicidad y duplicados ###
Países distintos: 179
Años distintos: 17
Filas duplicadas (exactamente iguales): 0

### Validación cruzada de columnas ###
No hay inconsistencias entre 'pais' y 'codigo_iso' en el DataFrame final.





### 3. Comparación regional: países latinoamericanos

En esta sección se busca identificar cuáles son los países de América Latina que han presentado los valores extremos del **Índice de Libertad de Prensa** en cada año observado.

> Recuerda que un menor puntaje en `indice` implica mayor libertad de prensa.

#### **Tareas:**

**a)** Utilizando un ciclo `for`, recorre cada año del conjunto de datos filtrado por países latinoamericanos, y determina para cada año:

* El país con el menor valor de `indice` (mayor libertad de prensa).
* El país con el mayor valor de `indice` (menor libertad de prensa).

**b)** Resuelve la misma tarea del punto anterior utilizando un enfoque vectorizado con `groupby`, sin usar ciclos explícitos.



#### **Lista de países latinoamericanos considerada:**

```python
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']
```

> Puedes usar esta lista para filtrar el DataFrame final por la columna `codigo_iso`.



In [8]:
# Task 3: Comparación regional: países latinoamericanos

america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
       'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
       'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
       'USA', 'VEN']

# Filter the DataFrame for Latin American countries
df_america = df[df['codigo_iso'].isin(america)]

print("### Task 3a: Finding min/max indice per year using a for loop ###")
results_for_loop = []
for anio_val in df_america['anio'].unique():
    df_year = df_america[df_america['anio'] == anio_val]
    # Drop rows where 'indice' is NaN before finding min/max
    df_year_cleaned = df_year.dropna(subset=['indice'])

    if not df_year_cleaned.empty:
        min_indice_country = df_year_cleaned.loc[df_year_cleaned['indice'].idxmin()]
        max_indice_country = df_year_cleaned.loc[df_year_cleaned['indice'].idxmax()]

        results_for_loop.append({
            'anio': anio_val,
            'pais_min_indice': min_indice_country['pais'],
            'indice_min': min_indice_country['indice'],
            'pais_max_indice': max_indice_country['pais'],
            'indice_max': max_indice_country['indice']
        })

results_df_for_loop = pd.DataFrame(results_for_loop).sort_values(by='anio').set_index('anio')
display(results_df_for_loop)

print("\n### Task 3b: Finding min/max indice per year using vectorized groupby ###")
# Drop rows with NaN in 'indice' before grouping to avoid issues with min/max calculation
df_america_cleaned = df_america.dropna(subset=['indice'])

# Find the index of the minimum and maximum 'indice' for each year
idx_min = df_america_cleaned.groupby('anio')['indice'].idxmin()
idx_max = df_america_cleaned.groupby('anio')['indice'].idxmax()

# Retrieve the rows corresponding to these indices
min_countries_grouped = df_america_cleaned.loc[idx_min, ['anio', 'pais', 'indice']].rename(columns={'pais': 'pais_min_indice', 'indice': 'indice_min'}).set_index('anio')
max_countries_grouped = df_america_cleaned.loc[idx_max, ['anio', 'pais', 'indice']].rename(columns={'pais': 'pais_max_indice', 'indice': 'indice_max'}).set_index('anio')

# Combine the results using merge on 'anio' (which is now the index)
results_df_groupby = pd.merge(min_countries_grouped, max_countries_grouped, left_index=True, right_index=True)
display(results_df_groupby)

### Task 3a: Finding min/max indice per year using a for loop ###


,pais_min_indice,indice_min,pais_max_indice,indice_max
anio,,,,
2001,Canadá,0.80,Cuba,90.30
2002,Trinidad y Tobago,1.00,Cuba,97.83
2003,Trinidad y Tobago,2.00,Argentina,35826.00
2004,Trinidad y Tobago,2.00,Cuba,87.00
2005,Bolivia,4.50,Cuba,95.00
2006,Canadá,4.88,Cuba,96.17
2007,Canadá,3.33,Cuba,88.33
2008,Canadá,3.70,Cuba,94.00
2009,Estados Unidos,6.75,Cuba,78.00



### Task 3b: Finding min/max indice per year using vectorized groupby ###


,pais_min_indice,indice_min,pais_max_indice,indice_max
anio,,,,
2001,Canadá,0.80,Cuba,90.30
2002,Trinidad y Tobago,1.00,Cuba,97.83
2003,Trinidad y Tobago,2.00,Argentina,35826.00
2004,Trinidad y Tobago,2.00,Cuba,87.00
2005,Bolivia,4.50,Cuba,95.00
2006,Canadá,4.88,Cuba,96.17
2007,Canadá,3.33,Cuba,88.33
2008,Canadá,3.70,Cuba,94.00
2009,Estados Unidos,6.75,Cuba,78.00


### 4. Análisis anual del índice por país

En esta sección se busca analizar la evolución del **índice máximo** de libertad de prensa alcanzado por cada país a lo largo del tiempo.

#### **Tarea principal:**

* Construye una tabla dinámica (`pivot_table`) donde las **filas** correspondan a los países, las **columnas** a los años (`anio`) y los **valores** sean el `indice` máximo alcanzado por cada país en ese año.
* Asegúrate de reemplazar los valores nulos resultantes con `0`.

> **Hint**: Puedes utilizar el parámetro `fill_value=0` en `pd.pivot_table(...)`.



#### **Preguntas adicionales:**

**a)** ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
**b)** ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?

> (Pista: usa `.mean(axis=0)` sobre la tabla pivot)

**c)** ¿Qué país muestra mayor **variabilidad** (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?

> (Pista: aplica `.max(axis=1) - .min(axis=1)`)

**d)** ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?

**e)** ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?





In [9]:
# Task 4: Análisis anual del índice por país

# Create the pivot table
pivot_df = pd.pivot_table(df, values='indice', index='pais', columns='anio', aggfunc='max', fill_value=0)
display(pivot_df.head())

print("\n### Preguntas adicionales sobre la tabla pivotante ###")

# a) ¿Qué país tiene el mayor valor de indice en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
max_indice_overall = pivot_df.max().max()
country_max_indice = pivot_df[pivot_df == max_indice_overall].stack().index[0][0]
print(f"a) País con el mayor 'indice' en toda la tabla: {country_max_indice} (Indice: {max_indice_overall})")

# To find the minimum non-zero value, we first replace 0s with NaN for calculation, then fill back to 0 for display if needed
min_indice_non_zero = pivot_df[pivot_df > 0].min().min()
country_min_indice = pivot_df[pivot_df == min_indice_non_zero].stack().index[0][0]
print(f"   País con el menor 'indice' (distinto de cero) en toda la tabla: {country_min_indice} (Indice: {min_indice_non_zero})")


# b) ¿Qué años presentan en promedio los valores de indice más altos? ¿Y los más bajos?
average_indice_by_year = pivot_df.mean(axis=0)
year_max_avg_indice = average_indice_by_year.idxmax()
max_avg_indice_value = average_indice_by_year.max()
year_min_avg_indice = average_indice_by_year.idxmin()
min_avg_indice_value = average_indice_by_year.min()

print(f"\nb) Año con el promedio de 'indice' más alto: {year_max_avg_indice} (Promedio: {max_avg_indice_value:.2f})")
print(f"   Año con el promedio de 'indice' más bajo: {year_min_avg_indice} (Promedio: {min_avg_indice_value:.2f})")


# c) ¿Qué país muestra mayor variabilidad (diferencia entre su máximo y mínimo indice a lo largo del tiempo)?
pivot_df_no_zeros = pivot_df.replace(0, np.nan) # Replace 0s with NaN to exclude them from min/max calculation
variability = pivot_df_no_zeros.max(axis=1) - pivot_df_no_zeros.min(axis=1)
country_max_variability = variability.idxmax()
max_variability_value = variability.max()

print(f"\nc) País con mayor variabilidad en 'indice': {country_max_variability} (Diferencia: {max_variability_value:.2f})")


# d) ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?
# A country has a constant index if max(indice) - min(indice) = 0 (excluding 0s)
# We need to consider only years where the index is not 0 for each country
constant_index_countries = variability[variability == 0].index.tolist()
if constant_index_countries:
    print(f"\nd) Países con 'indice' constante a lo largo de los años registrados: {constant_index_countries}")
else:
    print("\nd) No hay países con 'indice' constante a lo largo de los años registrados (excluyendo años con indice 0).")


# e) ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?
countries_all_zeros = pivot_df[(pivot_df == 0).all(axis=1)].index.tolist()
print(f"\ne) Países con todos los valores de 'indice' igual a 0: {countries_all_zeros}")
print("   Esto ocurre porque no tenían datos de 'indice' en el DataFrame original para ninguno de los años, o todos sus valores eran NaN, y fill_value=0 los convirtió a cero.")

anio,2001,2002,2003,2004,2005,2006,2007,2008,2009,2012,2013,2014,2015,2017,2018,2019
pais,,,,,,,,,,,,,,,,
Afghanistán,35.5,40.17,28.25,39.17,44.25,56.50,59.25,54.25,51.67,37.36,37.07,37.44,37.75,39.46,37.28,36.55
Albania,0.0,6.50,11.50,14.17,18.00,25.50,16.00,21.75,21.50,30.88,29.92,28.77,29.92,29.92,29.49,29.84
Alemania,1.5,1.33,2.00,4.00,5.50,5.75,4.50,3.50,4.25,10.24,10.23,11.47,14.80,14.97,14.39,14.60
Algeria,31.0,33.00,43.50,40.33,40.00,40.50,31.33,49.56,47.33,36.54,36.26,36.63,41.69,42.83,43.13,45.75
Andorra,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,6.82,6.82,19.87,19.87,21.03,22.21,24.63



### Preguntas adicionales sobre la tabla pivotante ###
a) País con el mayor 'indice' en toda la tabla: Kosovo (Indice: 64536.0)
   País con el menor 'indice' (distinto de cero) en toda la tabla: Austria (Indice: 0.5)

b) Año con el promedio de 'indice' más alto: 2013 (Promedio: 449.11)
   Año con el promedio de 'indice' más bajo: 2001 (Promedio: 20.03)

c) País con mayor variabilidad en 'indice': Kosovo (Diferencia: 46098.00)

d) Países con 'indice' constante a lo largo de los años registrados: ['Antigua y Barbuda', 'Granada']

e) Países con todos los valores de 'indice' igual a 0: []
   Esto ocurre porque no tenían datos de 'indice' en el DataFrame original para ninguno de los años, o todos sus valores eran NaN, y fill_value=0 los convirtió a cero.
